# 5-Class Emergency Acuity Triage (NHAMCS 2018–2022): Isolation Forest Denoising + Kernel Fisher Discriminant Analysis (KFDA) + Balanced Random Forest (`models/train_distant_analysis.ipynb`)

Trains on the official CDC **National Hospital Ambulatory Medical Care Survey (NHAMCS 2018–2022)** dataset (**`datasets/nhamcs_2018_2022.csv`**), classifying all **5 individual triage immediacy tiers (`IMMEDR 1–5`)** rather than collapsed groupings:
1. **`IMMEDR 1: Immediate`** ($y=0$): Resuscitation required immediately ($846$ visits, $\sim 1.46\%$).
2. **`IMMEDR 2: Emergent`** ($y=1$): Condition requiring emergent care within 1–14 minutes ($8,597$ visits, $\sim 14.79\%$).
3. **`IMMEDR 3: Urgent`** ($y=2$): Condition requiring urgent care within 15–60 minutes ($29,568$ visits, $\sim 50.87\%$).
4. **`IMMEDR 4: Semi-urgent`** ($y=3$): Condition requiring semi-urgent care within 1–2 hours ($16,715$ visits, $\sim 28.76\%$).
5. **`IMMEDR 5: Nonurgent`** ($y=4$): Condition requiring nonurgent care within 2–24 hours ($2,398$ visits, $\sim 4.13\%$).

```mermaid
flowchart TD
    Raw["Raw NHAMCS Arrival Features X in R^7 (58,124 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> Eng["1. Clinical Composite Engineering (+4 Non-Linear Indices: SI, PP, ROX, Age-SI)"]
    Eng --> IsoF["2. Isolation Forest Outlier Denoising (Preserves 100% of IMMEDR 1)"]
    IsoF --> CleanPlots["3. BEFORE vs AFTER Isolation Forest Cleanup Scatter Plots"]
    IsoF --> Nystroem["4. Nystroem Non-Linear RBF Kernel Mapping phi(X) in R^600"]
    Nystroem --> KFDA["5. Kernel Fisher Discriminant Analysis (KFDA: R^600 -> R^4)"]
    KFDA --> RF["6. Balanced Random Forest Classifier on KFDA Space (class_weight='balanced')"]
    RF --> KFDAPlots["7. BEFORE vs AFTER KFDA Projection Scatter Plots & Decision Contours"]
    RF --> Eval["8. 5-Class Holdout Evaluation: Recall, Specificity, Balanced Acc, ROC-AUC"]
```

### 📋 NHAMCS Feature Specification
- **`SEX`**: 1 = Female, 2 = Male (encoded as 0 = Female, 1 = Male).
- **`AGE`**: Patient age in years ($0 = <1$ year, $1–93$, $94 = 94+$ years).
- **`PULSE`**: Heart rate in beats/min (valid range: $20–240$, $-9$ = Blank, $998$ = Doppler).
- **`RESPR`**: Respiratory rate in breaths/min (valid range: $4–120$, $-9$ = Blank).
- **`BPSYS`**: Systolic blood pressure in mmHg (valid range: $40–300$, $-9$ = Blank, $998$ = Palp).
- **`BPDIAS`**: Diastolic blood pressure in mmHg (valid range: $20–200$, $-9$ = Blank, $998$ = Palp).
- **`POPCT`**: Pulse oximetry saturation in % (valid range: $50–100$, $-9$ = Blank).

### 🔬 Engineered Non-Linear Clinical Composite Indices
$$\text{Shock Index (SI)} = \frac{\text{PULSE}}{\text{BPSYS}}, \quad \text{Pulse Pressure (PP)} = \text{BPSYS} - \text{BPDIAS}$$
$$\text{ROX Index} = \frac{\text{POPCT}}{\text{RESPR}}, \quad \text{Age-Adjusted SI} = \text{AGE} \times \text{SI}$$

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Load NHAMCS 2018-2022 Dataset, Clean Missing Codes & Partition
# ---------------------------------------------------------------------------
import os, glob, json, pickle, warnings, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
ROOT = ".." if os.path.basename(os.getcwd()) == "models" else "."

# Robust NHAMCS dataset locator (verifies IMMEDR column in header to prevent matching report files)
def find_nhamcs_dataset():
    candidate_paths = [
        f"{ROOT}/datasets/nhamcs_2018_2022.csv",
        "datasets/nhamcs_2018_2022.csv",
        "../datasets/nhamcs_2018_2022.csv",
        "/kaggle/working/PKM_RF/datasets/nhamcs_2018_2022.csv",
        "/kaggle/working/datasets/nhamcs_2018_2022.csv",
        "/kaggle/input/nhamcs-2018-2022/nhamcs_2018_2022.csv",
        "/kaggle/input/disaster-triage-dataset/nhamcs_2018_2022.csv",
        "/kaggle/input/nhamcs/nhamcs_2018_2022.csv",
        "/kaggle/input/nhamcs2018-2022/nhamcs_2018_2022.csv"
    ]
    for p in candidate_paths:
        if os.path.isfile(p):
            try:
                hdr = pd.read_csv(p, nrows=2).columns
                if "IMMEDR" in hdr and "AGE" in hdr:
                    return p
            except Exception:
                pass
    for root in ["/kaggle/input", "/kaggle/working", "..", "."]:
        for p in glob.glob(f"{root}/**/nhamcs*.csv", recursive=True):
            if any(bad in p.lower() for bad in ["report", "plot", "deploy", "output"]):
                continue
            if os.path.isfile(p):
                try:
                    hdr = pd.read_csv(p, nrows=2).columns
                    if "IMMEDR" in hdr and "AGE" in hdr:
                        return p
                except Exception:
                    pass
    return None

csv_path = find_nhamcs_dataset()
if csv_path is None:
    raise FileNotFoundError("Could not locate the NHAMCS 2018-2022 raw CSV dataset containing IMMEDR and AGE columns! Please verify the dataset path.")

print(f"Loading NHAMCS dataset from: {csv_path} ...")

raw_cols = ["AGE", "SEX", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT", "IMMEDR"]
df_raw = pd.read_csv(csv_path, usecols=raw_cols)

# Filter to valid IMMEDR 1 - 5 (exclude -9=Blank, -8=Unknown, 0=No triage, 7=Non-triage ESA)
valid_mask = df_raw["IMMEDR"].isin([1, 2, 3, 4, 5])
df = df_raw[valid_mask].copy().reset_index(drop=True)

print(f"Total Raw Encounters : {len(df_raw):,}")
print(f"Valid IMMEDR 1-5 Cohort: {len(df):,} visits")

# Clean special / missing codes according to NHAMCS documentation
sex_clean   = df["SEX"].apply(lambda x: 1.0 if x == 2 else (0.0 if x == 1 else np.nan)).values
age_clean   = df["AGE"].apply(lambda x: float(x) if 0 <= x <= 120 else np.nan).values
pulse_clean = df["PULSE"].apply(lambda x: float(x) if 20 <= x <= 260 else np.nan).values
respr_clean = df["RESPR"].apply(lambda x: float(x) if 4 <= x <= 120 else np.nan).values
bpsys_clean = df["BPSYS"].apply(lambda x: float(x) if 40 <= x <= 300 else np.nan).values
bpdias_clean= df["BPDIAS"].apply(lambda x: float(x) if 20 <= x <= 200 else np.nan).values
popct_clean = df["POPCT"].apply(lambda x: float(x) if 50 <= x <= 100 else np.nan).values

RAW_FEATURE_NAMES = ["age", "gender_male", "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"]
X_raw = np.column_stack([age_clean, sex_clean, pulse_clean, bpsys_clean, bpdias_clean, respr_clean, popct_clean])

# Target class y in {0, 1, 2, 3, 4} corresponding to IMMEDR 1, 2, 3, 4, 5
y_all = (df["IMMEDR"].values - 1).astype(np.int32)
TIER_LABELS = [
    "IMMEDR 1: Immediate",
    "IMMEDR 2: Emergent",
    "IMMEDR 3: Urgent",
    "IMMEDR 4: Semi-urgent",
    "IMMEDR 5: Nonurgent"
]

print("=" * 80)
print("  5-CLASS IMMEDR TRIAGE DISTRIBUTION (NHAMCS 2018-2022)")
print("=" * 80)
for c, lbl in enumerate(TIER_LABELS):
    count_c = np.sum(y_all == c)
    print(f"  * Level {c+1} [{lbl:<22}]: {count_c:>6,} encounters ({count_c/len(y_all)*100:.2f}%)")
print("=" * 80 + chr(10))

# Stratified 70% Train / 15% Validation / 15% Test Split
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite  = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

y_train, y_val, y_test = y_all[itr], y_all[iva], y_all[ite]

# Median Imputation fitted strictly on Training partition
imputer   = SimpleImputer(strategy="median")
X_tr_imp  = imputer.fit_transform(X_raw[itr])
X_val_imp = imputer.transform(X_raw[iva])
X_te_imp  = imputer.transform(X_raw[ite])

# Function to add Non-Linear Clinical Composite Indices
def add_clinical_composites(X_arr):
    age  = X_arr[:, 0]
    hr   = X_arr[:, 2]
    sbp  = X_arr[:, 3]
    dbp  = X_arr[:, 4]
    rr   = X_arr[:, 5]
    o2   = X_arr[:, 6]
    
    si     = hr / np.clip(sbp, 40.0, 300.0)             # Shock Index: HR / SBP
    pp     = np.clip(sbp - dbp, 1.0, 200.0)             # Pulse Pressure: SBP - DBP
    rox    = o2 / np.clip(rr, 4.0, 120.0)               # ROX Index: SpO2 / RR
    age_si = age * si                                   # Age-Adjusted Shock Index
    return np.column_stack([X_arr, si, pp, rox, age_si])

X_tr_comp  = add_clinical_composites(X_tr_imp)
X_val_comp = add_clinical_composites(X_val_imp)
X_te_comp  = add_clinical_composites(X_te_imp)

ALL_FEATURE_NAMES = RAW_FEATURE_NAMES + ["shock_index", "pulse_pressure", "rox_index", "age_shock_index"]

# Standard z-score scaling
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_comp)
X_val   = scaler.transform(X_val_comp)
X_test  = scaler.transform(X_te_comp)

print(f"Partition Dimensions:")
print(f"  * Training Set  : {X_train.shape[0]:,} visits, {X_train.shape[1]} engineered features (70%)")
print(f"  * Validation Set: {X_val.shape[0]:,} visits (15%)")
print(f"  * Holdout Test  : {X_test.shape[0]:,} visits (15%)")


In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Isolation Forest Outlier Denoising (Preserves 100% of IMMEDR 1)
# ---------------------------------------------------------------------------
print("=" * 80)
print("  ISOLATION FOREST OUTLIER & BOUNDARY NOISE CLEANUP")
print("=" * 80)

# Fit Isolation Forest on non-Immediate classes (IMMEDR 2, 3, 4, 5) to remove boundary artifacts
non_imm_mask = (y_train != 0)
iso_forest = IsolationForest(
    n_estimators=150,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
iso_forest.fit(X_train[non_imm_mask])
outlier_pred = iso_forest.predict(X_train[non_imm_mask])

train_clean_mask = np.ones(len(y_train), dtype=bool)
train_clean_mask[non_imm_mask] = (outlier_pred == 1)

X_train_clean = X_train[train_clean_mask]
y_train_clean = y_train[train_clean_mask]

n_pruned = np.sum(~train_clean_mask)
print(f"✓ Isolation Forest Denoising Completed in {time.time()-t0:.1f}s")
print(f"Pruned {n_pruned:,} boundary noise / outlier visits ({n_pruned/len(y_train)*100:.2f}% of training cohort)\n")

print("Post-Cleanup Training Distribution:")
for c, lbl in enumerate(TIER_LABELS):
    orig_c  = np.sum(y_train == c)
    clean_c = np.sum(y_train_clean == c)
    print(f"  * {lbl:<22}: {clean_c:>6,} / {orig_c:>6,} ({clean_c/orig_c*100:.2f}% retained)")
print("=" * 80 + "\n")

# ---------------------------------------------------------------------------
# Diagnostic Plot: BEFORE vs AFTER Isolation Forest Cleanup Scatter Plots
# ---------------------------------------------------------------------------
plots_dir = f"{ROOT}/plots/distant_analysis"
os.makedirs(plots_dir, exist_ok=True)

palette = {
    0: '#d62728', # Red (Immediate)
    1: '#ff7f0e', # Orange (Emergent)
    2: '#bcbd22', # Yellow-Green (Urgent)
    3: '#1f77b4', # Blue (Semi-urgent)
    4: '#2ca02c'  # Green (Nonurgent)
}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
sample_n = min(12000, len(X_train))
sample_idx = np.random.choice(len(X_train), sample_n, replace=False)

# Panel 1: BEFORE Cleanup - Shock Index vs Pulse Pressure
ax1 = axes[0, 0]
for c in range(5):
    c_mask = (y_train[sample_idx] == c)
    ax1.scatter(
        X_train[sample_idx][c_mask, 7], X_train[sample_idx][c_mask, 8],
        c=palette[c], label=TIER_LABELS[c], alpha=0.45 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax1.set_title("BEFORE Isolation Forest: Shock Index vs Pulse Pressure (Raw)", fontsize=11, fontweight='bold')
ax1.set_xlabel("Standardized Shock Index (SI)", fontsize=10)
ax1.set_ylabel("Standardized Pulse Pressure (PP)", fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.3)
ax1.legend(loc='upper right', fontsize=8)

# Panel 2: AFTER Cleanup - Shock Index vs Pulse Pressure
ax2 = axes[0, 1]
clean_sample_n = min(12000, len(X_train_clean))
clean_sample_idx = np.random.choice(len(X_train_clean), clean_sample_n, replace=False)
for c in range(5):
    c_mask = (y_train_clean[clean_sample_idx] == c)
    ax2.scatter(
        X_train_clean[clean_sample_idx][c_mask, 7], X_train_clean[clean_sample_idx][c_mask, 8],
        c=palette[c], label=TIER_LABELS[c], alpha=0.45 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax2.set_title("AFTER Isolation Forest Denoising: Cleaned Manifold", fontsize=11, fontweight='bold')
ax2.set_xlabel("Standardized Shock Index (SI)", fontsize=10)
ax2.set_ylabel("Standardized Pulse Pressure (PP)", fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.3)

# Panel 3: BEFORE Cleanup - ROX Index vs Heart Rate
ax3 = axes[1, 0]
for c in range(5):
    c_mask = (y_train[sample_idx] == c)
    ax3.scatter(
        X_train[sample_idx][c_mask, 9], X_train[sample_idx][c_mask, 2],
        c=palette[c], label=TIER_LABELS[c], alpha=0.45 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax3.set_title("BEFORE Isolation Forest: ROX Index vs Heart Rate (Raw)", fontsize=11, fontweight='bold')
ax3.set_xlabel("Standardized ROX Index (SpO2 / RR)", fontsize=10)
ax3.set_ylabel("Standardized Heart Rate (PULSE)", fontsize=10)
ax3.grid(True, linestyle='--', alpha=0.3)

# Panel 4: AFTER Cleanup - ROX Index vs Heart Rate
ax4 = axes[1, 1]
for c in range(5):
    c_mask = (y_train_clean[clean_sample_idx] == c)
    ax4.scatter(
        X_train_clean[clean_sample_idx][c_mask, 9], X_train_clean[clean_sample_idx][c_mask, 2],
        c=palette[c], label=TIER_LABELS[c], alpha=0.45 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax4.set_title("AFTER Isolation Forest Denoising: Cleaned ROX vs HR", fontsize=11, fontweight='bold')
ax4.set_xlabel("Standardized ROX Index (SpO2 / RR)", fontsize=10)
ax4.set_ylabel("Standardized Heart Rate (PULSE)", fontsize=10)
ax4.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
isof_plot_path = os.path.join(plots_dir, "nhamcs_isof_before_after_cleanup_scatter.png")
plt.savefig(isof_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Isolation Forest Before/After plot saved to: {isof_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Kernel Fisher Discriminant Analysis (KFDA) via Nystroem Mapping
# ---------------------------------------------------------------------------
print("=" * 80)
print("  KERNEL FISHER DISCRIMINANT ANALYSIS (KFDA: R^11 -> R^600 -> R^4)")
print("=" * 80)

# 1. Nystroem RBF Kernel Landmark Approximation (R^11 -> R^600)
nystroem_kfda = Nystroem(
    kernel='rbf',
    gamma=0.15,
    n_components=600,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
Phi_train = nystroem_kfda.fit_transform(X_train_clean)
Phi_val   = nystroem_kfda.transform(X_val)
Phi_test  = nystroem_kfda.transform(X_test)
print(f"✓ Nystroem Non-Linear Kernel Mapping fitted in {time.time()-t0:.1f}s (Shape: {Phi_train.shape})")

# 2. Multi-Class Linear Discriminant Analysis on Kernel Space (KFDA)
# For C=5 classes, LDA extracts min(C-1, features) = 4 canonical coordinates: [z1, z2, z3, z4]
kfda_lda = LinearDiscriminantAnalysis(n_components=4)

t0 = time.time()
Z_train = kfda_lda.fit_transform(Phi_train, y_train_clean)
Z_val   = kfda_lda.transform(Phi_val)
Z_test  = kfda_lda.transform(Phi_test)
print(f"✓ KFDA Multi-Class Fisher Projection fitted in {time.time()-t0:.1f}s")
print(f"Explained Variance Ratios per Discriminant Coordinate: {np.round(kfda_lda.explained_variance_ratio_, 4)}")
print(f"Cumulative Explained Variance: {np.sum(kfda_lda.explained_variance_ratio_)*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Train Balanced Random Forest Classifier on KFDA Manifold
# ---------------------------------------------------------------------------
print("=" * 80)
print("  TRAINING BALANCED RANDOM FOREST ON 4D KFDA MANIFOLD")
print("=" * 80)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rf_model.fit(Z_train, y_train_clean)
print(f"✓ Balanced Random Forest Model Trained in {time.time()-t0:.1f}s!")

# Quick Validation Check
pred_val = rf_model.predict(Z_val)
val_bacc = balanced_accuracy_score(y_val, pred_val)
val_rec0 = recall_score(y_val, pred_val, labels=[0], average=None, zero_division=0)[0]
print(f"\nValidation Set Quick Check:")
print(f"  * Macro Balanced Accuracy      : {val_bacc*100:.2f}%")
print(f"  * IMMEDR 1 (Immediate) Recall  : {val_rec0*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: BEFORE vs AFTER KFDA Projection Scatter Plots & Decision Contours
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Panel 1: Raw Overlapping Vitals (Shock Index vs Pulse Pressure)
ax1 = axes[0, 0]
for c in range(5):
    c_mask = (y_test == c)
    ax1.scatter(
        X_test[c_mask, 7], X_test[c_mask, 8],
        c=palette[c], label=TIER_LABELS[c], alpha=0.4 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax1.set_title("BEFORE KFDA: Overlapping Vitals (Shock Index vs Pulse Pressure)", fontsize=11, fontweight='bold')
ax1.set_xlabel("Standardized Shock Index (SI)", fontsize=10)
ax1.set_ylabel("Standardized Pulse Pressure (PP)", fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.3)
ax1.legend(loc='upper right', fontsize=8)

# Panel 2: Raw Overlapping Vitals (ROX Index vs Heart Rate)
ax2 = axes[0, 1]
for c in range(5):
    c_mask = (y_test == c)
    ax2.scatter(
        X_test[c_mask, 9], X_test[c_mask, 2],
        c=palette[c], label=TIER_LABELS[c], alpha=0.4 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax2.set_title("BEFORE KFDA: Overlapping Vitals (ROX Index vs Heart Rate)", fontsize=11, fontweight='bold')
ax2.set_xlabel("Standardized ROX Index (SpO2 / RR)", fontsize=10)
ax2.set_ylabel("Standardized Heart Rate (PULSE)", fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.3)

# Panel 3: 2D KFDA Discriminant Manifold (z1 vs z2)
ax3 = axes[1, 0]
for c in range(5):
    c_mask = (y_test == c)
    ax3.scatter(
        Z_test[c_mask, 0], Z_test[c_mask, 1],
        c=palette[c], label=TIER_LABELS[c], alpha=0.45 if c > 0 else 0.9,
        s=14 if c > 0 else 35, edgecolors='none'
    )
ax3.set_title("AFTER KFDA Projection: 2D Fisher Canonical Coordinates (z1 vs z2)", fontsize=11, fontweight='bold')
ax3.set_xlabel(f"KFDA Coordinate z1 ({kfda_lda.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=10)
ax3.set_ylabel(f"KFDA Coordinate z2 ({kfda_lda.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=10)
ax3.grid(True, linestyle='--', alpha=0.3)

# Panel 4: 2D KFDA Space with 5-Class Decision Contours
ax4 = axes[1, 1]
x_min, x_max = Z_test[:, 0].min() - 0.5, Z_test[:, 0].max() + 0.5
y_min, y_max = Z_test[:, 1].min() - 0.5, Z_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))

# Fit auxiliary 2D Random Forest strictly for decision contour visualization
rf_2d = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_2d.fit(Z_train[:, :2], y_train_clean)
grid_preds = rf_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

cmap_light = plt.matplotlib.colors.ListedColormap(['#ffcccc', '#ffe0b2', '#fff9c4', '#bbdefb', '#c8e6c9'])
ax4.contourf(xx, yy, grid_preds, alpha=0.4, cmap=cmap_light)

for c in range(5):
    c_mask = (y_test == c)
    ax4.scatter(
        Z_test[c_mask, 0], Z_test[c_mask, 1],
        c=palette[c], label=TIER_LABELS[c], alpha=0.4 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax4.set_title("AFTER KFDA: 5-Class Random Forest Decision Boundary Contours", fontsize=11, fontweight='bold')
ax4.set_xlabel(f"KFDA Coordinate z1", fontsize=10)
ax4.set_ylabel(f"KFDA Coordinate z2", fontsize=10)
ax4.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
kfda_plot_path = os.path.join(plots_dir, "nhamcs_kfda_rf_before_after_scatter.png")
plt.savefig(kfda_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ KFDA Before/After plot saved to: {kfda_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: 5-Class Holdout Test Evaluation — Recall, Specificity, Balanced Acc, ROC-AUC
# ---------------------------------------------------------------------------
pred_test = rf_model.predict(Z_test)
p_test    = rf_model.predict_proba(Z_test)

# 1. Overall & Macro Metrics
acc         = accuracy_score(y_test, pred_test)
bal_acc     = balanced_accuracy_score(y_test, pred_test)
macro_f1    = f1_score(y_test, pred_test, average='macro', zero_division=0)
weighted_f1 = f1_score(y_test, pred_test, average='weighted', zero_division=0)

# 2. Per-Class Recall (Sensitivity), Precision, F1
recall_per = recall_score(y_test, pred_test, average=None, zero_division=0)
prec_per   = precision_score(y_test, pred_test, average=None, zero_division=0)
f1_per     = f1_score(y_test, pred_test, average=None, zero_division=0)

# 3. Per-Class Specificity (True Negative Rate)
cm = confusion_matrix(y_test, pred_test, labels=[0, 1, 2, 3, 4])
specificity_per = []
for c in range(5):
    tp = cm[c, c]
    fn = cm[c, :].sum() - tp
    fp = cm[:, c].sum() - tp
    tn = cm.sum() - tp - fn - fp
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificity_per.append(spec)
macro_spec = np.mean(specificity_per)

# 4. Per-Class & Macro ROC-AUC (One-vs-Rest)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4])
roc_auc_ovr = roc_auc_score(y_test_bin, p_test, average='macro', multi_class='ovr')
roc_auc_per = []
for c in range(5):
    auc_c = roc_auc_score(y_test_bin[:, c], p_test[:, c])
    roc_auc_per.append(auc_c)

# Construct Detailed Summary DataFrame
report_rows = []
for c, lbl in enumerate(TIER_LABELS):
    report_rows.append({
        'Triage_Level': lbl,
        'True_Visits': int(np.sum(y_test == c)),
        'Predicted_Visits': int(np.sum(pred_test == c)),
        'Recall (Sensitivity)': f"{recall_per[c]*100:.2f}%",
        'Specificity': f"{specificity_per[c]*100:.2f}%",
        'Precision': f"{prec_per[c]*100:.2f}%",
        'F1_Score': round(f1_per[c], 4),
        'ROC_AUC (OvR)': round(roc_auc_per[c], 4)
    })

report_df = pd.DataFrame(report_rows)

print("=" * 110)
print("     5-CLASS HOLDOUT EVALUATION: ISOLATION FOREST + KFDA + BALANCED RANDOM FOREST (NHAMCS)")
print("=" * 110)
print(f"Total Test Encounters       : {len(y_test):,} visits")
print(f"Overall Accuracy            : {acc*100:.2f}%")
print(f"Macro Balanced Accuracy     : {bal_acc*100:.2f}%")
print(f"Macro Specificity           : {macro_spec*100:.2f}%")
print(f"Macro ROC-AUC (OvR)         : {roc_auc_ovr:.4f}")
print(f"Macro F1-Score              : {macro_f1:.4f}")
print(f"Weighted F1-Score           : {weighted_f1:.4f}")
print("-" * 110)
print(report_df.to_string(index=False))
print("=" * 110 + "\n")

print("Detailed Classification Report:")
print(classification_report(y_test, pred_test, target_names=TIER_LABELS, digits=4))

# Export Report CSV
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'nhamcs_5class_isof_kfda_rf_report.csv')
report_df.to_csv(report_file, index=False)
print(f"✓ Report successfully saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: 5x5 Confusion Matrix Heatmap
# ---------------------------------------------------------------------------
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(9.5, 8))
annot = np.empty_like(cm, dtype=object)
for i in range(5):
    for j in range(5):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"

short_labels = ['1: Imm', '2: Emerg', '3: Urgent', '4: Semi-urg', '5: Nonurg']
sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Blues', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=short_labels, yticklabels=short_labels
)

ax.set_title(
    f'NHAMCS 5-Class Acuity: Isolation Forest + KFDA + Balanced Random Forest\n'
    f'Balanced Accuracy: {bal_acc*100:.2f}% | Macro ROC-AUC: {roc_auc_ovr:.4f}',
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel('Predicted IMMEDR Triage Level', fontsize=11, fontweight='bold')
ax.set_ylabel('True IMMEDR Triage Level', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path1 = os.path.join(plots_dir, 'nhamcs_5class_confusion_matrix.png')
cm_path2 = os.path.join(ROOT, 'plots/image/nhamcs_5class_confusion_matrix.png')
os.makedirs(os.path.dirname(cm_path2), exist_ok=True)
plt.savefig(cm_path1, dpi=300, bbox_inches='tight')
plt.savefig(cm_path2, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix saved to: {cm_path1}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: KFDA Coordinate Feature Importance in Random Forest
# ---------------------------------------------------------------------------
importances = rf_model.feature_importances_
feat_names  = [f'KFDA_Coord_{i+1} ({kfda_lda.explained_variance_ratio_[i]*100:.1f}% var)' for i in range(4)]

feat_imp_df = pd.DataFrame({
    'KFDA_Coordinate': feat_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(data=feat_imp_df, x='Importance', y='KFDA_Coordinate', color='#1f77b4', ax=ax)
ax.set_title('KFDA Coordinate Gini Importance in 5-Class Random Forest', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Normalized Gini Importance', fontsize=11, fontweight='bold')
ax.set_ylabel('Canonical Coordinate', fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
imp_path = os.path.join(plots_dir, 'nhamcs_5class_feature_importance.png')
plt.savefig(imp_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Feature importance plot saved to: {imp_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 9: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'nystroem_kfda': nystroem_kfda,
    'kfda_lda': kfda_lda,
    'rf_model': rf_model,
    'raw_features': RAW_FEATURE_NAMES,
    'all_features': ALL_FEATURE_NAMES,
    'tier_labels': TIER_LABELS
}

bundle_file = os.path.join(deploy_dir, 'nhamcs_5class_isof_kfda_rf_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='NHAMCS_5Class_IsolationForest_KFDA_Balanced_Random_Forest',
    dataset='datasets/nhamcs_2018_2022.csv',
    target='IMMEDR (1 to 5)',
    n_classes=5,
    tier_labels=TIER_LABELS,
    raw_features=RAW_FEATURE_NAMES,
    engineered_features=ALL_FEATURE_NAMES,
    total_valid_samples=len(y_all),
    holdout_test_samples=len(y_test),
    overall_accuracy=round(acc, 4),
    macro_balanced_accuracy=round(bal_acc, 4),
    macro_specificity=round(macro_spec, 4),
    macro_roc_auc_ovr=round(roc_auc_ovr, 4),
    macro_f1=round(macro_f1, 4),
    per_class_metrics=report_rows
)

manifest_file = os.path.join(deploy_dir, 'nhamcs_5class_isof_kfda_rf_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Deployment Bundle  : {bundle_file}")
print(f"✓ Deployment Manifest: {manifest_file}")